# 02 — SVM v1 (eight classes)

The paper lists **nine** species. Drive has no *Thalassia hemprichii*, so this run trains the **eight** classes that exist, using the **`all`** folders only.

Skipped: research-grade, any `- Copy` folder, *Halophila spinulosa*, *Thalassia*.

**CPU is enough** (SVM does not need a GPU). This run is a **smoke test**: `MAX_PER_CLASS = 100` (~700 images). Fast enough to see if SVM training works. Do not treat the accuracy as the study’s 85% result.

Raise to `250` or `None` (all ~3896) for a real v1 after the test run looks sane. GLCM is on 64px.

Progress bars appear on folder scan, image load, and feature extract. A cell sitting at 0% on Drive is still working.

## A. Mount Drive

Same account as the `Seagrass` shortcut. If mount fails: Files → Mount Drive, then re-run.

In [ ]:
from pathlib import Path
import json
import sys

IN_COLAB = Path("/content").exists()
print("colab", IN_COLAB)

DRIVE_MOUNTED = Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists()
if IN_COLAB and not DRIVE_MOUNTED:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=True)
print("drive ok")

In [ ]:
CANDIDATE_ROOTS = [
    Path("/content/drive/MyDrive/Seagrass"),
    Path("/content/drive/My Drive/Seagrass"),
]
MARKER = "cymodocea rotundata"

def looks_like_dataset_root(path: Path) -> bool:
    if not path.is_dir():
        return False
    names = " ".join(p.name.lower() for p in path.iterdir() if p.is_dir())
    return MARKER in names

def find_root() -> Path:
    for cand in CANDIDATE_ROOTS:
        if looks_like_dataset_root(cand):
            return cand
    for base in (Path("/content/drive/MyDrive"), Path("/content/drive/My Drive")):
        if not base.is_dir():
            continue
        for path in base.rglob("*"):
            if path.is_dir() and looks_like_dataset_root(path):
                return path
    raise FileNotFoundError("Seagrass dataset not found under My Drive")

DATA_ROOT = find_root()
print("DATA_ROOT", DATA_ROOT)

## B. Pipeline

Uses `ml/src` if this repo is on the machine. Otherwise the same functions are defined in the next cell.

In [ ]:
import time
import numpy as np
from PIL import Image, ImageOps
from collections import Counter
from tqdm.auto import tqdm

def step(title: str):
    print(f"\n>>> {title}", flush=True)

STUDY_SPECIES = (
    "cymodocea_rotundata", "cymodocea_serrulata", "enhalus_acoroides",
    "halodule_pinifolia", "halodule_uninervis", "halophila_minor",
    "halophila_ovalis", "syringodium_isoetifolium", "thalassia_hemprichii",
)
HELD_OUT_SPECIES = ("thalassia_hemprichii",)
TRAINABLE_SPECIES = tuple(s for s in STUDY_SPECIES if s not in HELD_OUT_SPECIES)
TRAIN_VARIANT = "all"
IMAGE_SIZE = 256
SPLIT_SEED = 42
IMAGE_SUFFIXES = (".jpg", ".jpeg", ".png", ".webp", ".bmp", ".tif", ".tiff")
SPECIES_PHRASES = {s: s.replace("_", " ") for s in STUDY_SPECIES}
OUT_OF_SCOPE_PHRASES = {"halophila_spinulosa": "halophila spinulosa"}

def classify_folder(name: str):
    n = " ".join(name.lower().replace("_", " ").split())
    variant = "unknown"
    if "copy" in n:
        variant = "copy"
    elif "research-grade" in n or "research grade" in n:
        variant = "research-grade"
    elif "dataset (all)" in n or " (all)" in n or n.endswith("(all)"):
        variant = "all"
    elif "dataset" in n:
        variant = "all"
    for slug, phrase in SPECIES_PHRASES.items():
        if phrase in n:
            return slug, "study", variant
    for slug, phrase in OUT_OF_SCOPE_PHRASES.items():
        if phrase in n:
            return slug, "extra", variant
    return None, "unknown", variant

print("trainable", len(TRAINABLE_SPECIES), TRAINABLE_SPECIES)

In [ ]:
def list_labeled_images(root: Path):
    folders = []
    for child in sorted(root.iterdir(), key=lambda p: p.name.lower()):
        if not child.is_dir() or child.name.startswith("."):
            continue
        slug, scope, variant = classify_folder(child.name)
        if scope != "study" or slug not in TRAINABLE_SPECIES or variant != TRAIN_VARIANT:
            continue
        folders.append((child, slug))
    pairs = []
    for child, slug in tqdm(folders, desc="Scan Drive folders", unit="folder"):
        n_before = len(pairs)
        for path in child.rglob("*"):
            if path.is_file() and path.suffix.lower() in IMAGE_SUFFIXES:
                pairs.append((path, slug))
        tqdm.write(f"  {slug}: {len(pairs) - n_before} images in {child.name}")
    return pairs

step("Listing images on Drive")
t0 = time.perf_counter()
pairs = list_labeled_images(DATA_ROOT)
print(f"listed in {time.perf_counter() - t0:.1f}s")
print("images on Drive (all folders)", len(pairs))
print(Counter(slug for _, slug in pairs))

# Smoke test: 100/class. Raise to 250 or None after this run works.
MAX_PER_CLASS = 100
if MAX_PER_CLASS is not None:
    rng = np.random.default_rng(SPLIT_SEED)
    capped = []
    by = {}
    for path, slug in pairs:
        by.setdefault(slug, []).append((path, slug))
    for slug, items in by.items():
        if len(items) > MAX_PER_CLASS:
            pick = rng.choice(len(items), size=MAX_PER_CLASS, replace=False)
            items = [items[int(i)] for i in pick]
        capped.extend(items)
    pairs = capped

print("images this run", len(pairs), "max_per_class", MAX_PER_CLASS)
print(Counter(slug for _, slug in pairs))
assert set(Counter(slug for _, slug in pairs)) == set(TRAINABLE_SPECIES), "a trainable class is empty"

## C. Split, load, augment train

70 / 15 / 15 stratified. Augment only the train split for classes under 80 images.

In [ ]:
from sklearn.model_selection import train_test_split

paths = [p for p, _ in pairs]
labels = [s for _, s in pairs]
temp_p, test_p, temp_y, test_y = train_test_split(
    paths, labels, test_size=0.15, stratify=labels, random_state=SPLIT_SEED
)
train_p, val_p, train_y, val_y = train_test_split(
    temp_p, temp_y, test_size=0.15 / 0.85, stratify=temp_y, random_state=SPLIT_SEED
)
print("split", len(train_p), len(val_p), len(test_p))

def load_rgb(path, size=IMAGE_SIZE):
    image = Image.open(path)
    image = ImageOps.exif_transpose(image).convert("RGB")
    return np.asarray(image.resize((size, size), Image.Resampling.LANCZOS), dtype=np.uint8)

def load_many(ps, ys, tag):
    images = []
    keep_y = []
    t0 = time.perf_counter()
    for path, y in tqdm(list(zip(ps, ys)), desc=f"Load {tag}", unit="img"):
        try:
            images.append(load_rgb(path))
            keep_y.append(y)
        except Exception as exc:
            tqdm.write(f"skip {path}: {exc}")
    print(f"{tag} done {len(images)} in {time.perf_counter() - t0:.1f}s")
    return images, keep_y

step("Loading images (Drive can sit on 0% then jump)")
train_img, train_y = load_many(train_p, train_y, "train")
val_img, val_y = load_many(val_p, val_y, "val")
test_img, test_y = load_many(test_p, test_y, "test")

In [ ]:
def augment_rgb(rgb, kind):
    image = Image.fromarray(rgb)
    if kind == "hflip":
        image = ImageOps.mirror(image)
    elif kind == "rot90":
        image = image.rotate(90, expand=False)
    elif kind == "rot270":
        image = image.rotate(270, expand=False)
    elif kind == "zoom":
        w, h = image.size
        inset = int(min(w, h) * 0.1)
        image = image.crop((inset, inset, w - inset, h - inset)).resize((w, h), Image.Resampling.LANCZOS)
    return np.asarray(image, dtype=np.uint8)

def expand_minority(images, labels, min_per_class=80):
    kinds = ("hflip", "rot90", "rot270", "zoom")
    out_i, out_y = list(images), list(labels)
    by = {}
    for im, lab in zip(images, labels):
        by.setdefault(lab, []).append(im)
    for lab, items in by.items():
        need = max(0, min_per_class - len(items))
        idx = 0
        while idx < need:
            out_i.append(augment_rgb(items[idx % len(items)], kinds[idx % len(kinds)]))
            out_y.append(lab)
            idx += 1
    return out_i, out_y

step("Augment minority train classes")
print("train before", Counter(train_y))
train_img, train_y = expand_minority(train_img, train_y)
print("train after ", Counter(train_y))

## D. Features + SVM

HSV histogram, GLCM texture, largest-blob shape. Then StandardScaler + RBF SVM, class_weight=balanced.

In [ ]:
from skimage.color import rgb2gray, rgb2hsv
from skimage.feature import graycomatrix, graycoprops
from skimage.filters import threshold_otsu
from skimage.measure import label, regionprops

from skimage.transform import resize

BINS, LEVELS, GLCM_SIZE = 16, 32, 64
DISTANCES = (1, 2)
ANGLES = (0.0, np.pi / 4, np.pi / 2, 3 * np.pi / 4)
GLCM_PROPS = ("contrast", "homogeneity", "energy", "correlation")

def extract_vector(rgb):
    hsv = rgb2hsv(rgb.astype(np.float64) / 255.0)
    color = np.concatenate([
        np.histogram(hsv[:, :, c], bins=BINS, range=(0.0, 1.0), density=True)[0]
        for c in range(3)
    ])
    gray = (rgb2gray(rgb) * 255.0).astype(np.uint8)
    small = resize(gray, (GLCM_SIZE, GLCM_SIZE), preserve_range=True, anti_aliasing=True).astype(np.uint8)
    q = np.clip(small.astype(np.uint16) * (LEVELS - 1) // 255, 0, LEVELS - 1).astype(np.uint8)
    glcm = graycomatrix(q, distances=DISTANCES, angles=ANGLES, levels=LEVELS, symmetric=True, normed=True)
    texture = np.concatenate([graycoprops(glcm, p).ravel() for p in GLCM_PROPS])
    shape = np.zeros(6, dtype=np.float64)
    try:
        mask = gray < threshold_otsu(gray)
        lab = label(mask)
        if lab.max() == 0:
            lab = label(~mask)
        if lab.max() > 0:
            r = max(regionprops(lab), key=lambda x: x.area)
            y0, x0, y1, x1 = r.bbox
            circ = 0.0 if r.perimeter <= 0 else float(4 * np.pi * r.area / (r.perimeter ** 2))
            shape = np.array([
                r.area / gray.size,
                r.perimeter / (gray.shape[0] + gray.shape[1]),
                circ,
                max(x1 - x0, 1) / max(y1 - y0, 1),
                float(r.solidity),
                float(r.extent),
            ], dtype=np.float64)
    except ValueError:
        pass
    return np.concatenate([color, texture, shape])

def extract_batch(images, tag):
    rows = []
    t0 = time.perf_counter()
    for im in tqdm(images, desc=f"Features {tag}", unit="img"):
        rows.append(extract_vector(im))
    print(f"feat {tag} done in {time.perf_counter() - t0:.1f}s")
    return np.stack(rows, axis=0)

step("Extracting color / GLCM / shape features")
X_train = extract_batch(train_img, "train")
X_val = extract_batch(val_img, "val")
X_test = extract_batch(test_img, "test")
print(X_train.shape, X_val.shape, X_test.shape)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt

model = Pipeline([
    ("scaler", StandardScaler()),
    ("svm", SVC(kernel="rbf", C=10.0, gamma="scale", class_weight="balanced",
                probability=True, random_state=SPLIT_SEED)),
])
step("Fitting SVM (no bar — sklearn is one call, usually 1–3 min)")
t0 = time.perf_counter()
model.fit(X_train, train_y)
print(f"SVM fitted in {time.perf_counter() - t0:.1f}s")

def show_split(name, X, y):
    pred = model.predict(X)
    acc = accuracy_score(y, pred)
    print(f"\n{name} accuracy {acc:.4f}")
    print(classification_report(y, pred, labels=list(TRAINABLE_SPECIES), zero_division=0))
    return acc, pred

val_acc, _ = show_split("val", X_val, val_y)
test_acc, test_pred = show_split("test", X_test, test_y)

cm = confusion_matrix(test_y, test_pred, labels=list(TRAINABLE_SPECIES))
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm)
ax.set_xticks(range(len(TRAINABLE_SPECIES)))
ax.set_yticks(range(len(TRAINABLE_SPECIES)))
ax.set_xticklabels(TRAINABLE_SPECIES, rotation=75, ha="right", fontsize=8)
ax.set_yticklabels(TRAINABLE_SPECIES, fontsize=8)
ax.set_title(f"test confusion  acc={test_acc:.3f}")
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()
print("study target is 85% test accuracy. this run:", round(test_acc * 100, 2), "%")

## E. Save artifacts

Writes to Drive `Seagrass-ml-artifacts/v1/`. Download those files into the repo `ml/artifacts/` later.

In [ ]:
import joblib
from sklearn.metrics import classification_report as _cr

dest = Path("/content/drive/MyDrive/Seagrass-ml-artifacts/v1")
dest.mkdir(parents=True, exist_ok=True)
joblib.dump(model, dest / "svm_seagrass_v1.joblib")
metrics = {
    "classes": list(TRAINABLE_SPECIES),
    "held_out": list(HELD_OUT_SPECIES),
    "variant": TRAIN_VARIANT,
    "image_size": IMAGE_SIZE,
    "n_train": len(train_y),
    "n_val": len(val_y),
    "n_test": len(test_y),
    "val_accuracy": float(val_acc),
    "test_accuracy": float(test_acc),
    "test_report": _cr(test_y, test_pred, labels=list(TRAINABLE_SPECIES), output_dict=True, zero_division=0),
    "confusion_matrix": cm.tolist(),
}
(dest / "metrics.json").write_text(json.dumps(metrics, indent=2), encoding="utf-8")
(dest / "label_map.json").write_text(
    json.dumps({str(i): s for i, s in enumerate(TRAINABLE_SPECIES)}, indent=2), encoding="utf-8"
)
print("wrote", dest)
print("test_accuracy", metrics["test_accuracy"])